In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from moc.configs.config import get_config
from moc.utils.run_config import RunConfig
# from moc.models.mqf2.lightning_module import MQF2LightningModule
# from moc.models.mixture.mixture_model2 import MixtureLightningModule
from moc.models.gaussian.gaussian import GaussianLightningModule
from moc.models.trainers.lightning_trainer import get_lightning_trainer
from moc.datamodules.generated import GeneratedDataModule
import numpy as np
import matplotlib.pyplot as plt
plt.style.use('seaborn-v0_8')
from moc.metrics.distribution_metrics import pce
import torch
from torch.distributions import MultivariateNormal

/home/naomi/miniconda/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
torch.manual_seed(42)

In [4]:
def build_multivariate_normal(d, sigma2, tau, mean):
    indices = torch.arange(d).unsqueeze(0)
    covariance_matrix = sigma2 * torch.exp(-torch.abs(indices.T - indices) / tau)
    return MultivariateNormal(mean, covariance_matrix)

In [5]:
d = 10
sigma2 = 1.0
tau = 1.0

mean = torch.zeros(d)
true_dist = build_multivariate_normal(d, sigma2, tau, mean)
# undercorr = build_multivariate_normal(d, sigma2, tau, mean)

In [6]:
undermean = torch.ones(d) * - 0.5
overmean = torch.ones(d) * 0.5
undermean_dist = build_multivariate_normal(d, sigma2, tau, undermean)
overmean_dist = build_multivariate_normal(d, sigma2, tau, overmean)
undervar_dist = build_multivariate_normal(d, 0.85, tau, mean)
overvar_dist = build_multivariate_normal(d, 1.25, tau, mean)
undercorr_dist = build_multivariate_normal(d, sigma2, 0.5, mean)
overcorr_dist = build_multivariate_normal(d, sigma2, 2, mean)


In [7]:
N = 10000
y = true_dist.sample((N,))
y.shape

torch.Size([10000, 10])

In [11]:
pces, cdf, var = pce(overcorr_dist, y, n_samples = 100, mode = 'all', prerank = 'cdf', setup='simulated')
print(pces)

1
tensor([0.1830])


In [14]:
def print_pce_for_dist(name, dist):
    pces, cdf, var = pce(dist, y, n_samples=100, mode='all', prerank='cdf', setup='simulated')
    print(f"{name} PCEs: {pces}")

In [15]:
print_pce_for_dist("Undermean", undermean_dist)
print_pce_for_dist("Overmean", overmean_dist)
print_pce_for_dist("Undervariance", undervar_dist)
print_pce_for_dist("Overvariance", overvar_dist)
print_pce_for_dist("Undercorrelation", undercorr_dist)
print_pce_for_dist("Overcorrelation", overcorr_dist)

1
Undermean PCEs: tensor([0.4900])
1
Overmean PCEs: tensor([0.4900])
1
Undervariance PCEs: tensor([0.1527])
1
Overvariance PCEs: tensor([0.1385])
1
Undercorrelation PCEs: tensor([0.2089])
1
Overcorrelation PCEs: tensor([0.1641])


In [ ]:
#np.sum(pces*var)

In [ ]:
config = get_config()
config.device = 'cpu'

In [ ]:
seeds = [42]
prerank = 'marginal'
for seed in seeds:
    rc = RunConfig(config, 'synthetic', 'linear_case', seed = seed)
    datamodule = GeneratedDataModule(rc, seed=seed, num_workers=8)
    p, q = datamodule.input_dim, datamodule.output_dim
    model = GaussianLightningModule(p, q, prerank = prerank)
    # model = MixtureLightningModule(p, q, prerank = prerank)
    #model = MQF2LightningModule(p, q)
    trainer = get_lightning_trainer(rc)
    trainer.fit(model, datamodule)
    # wandb.finish()
    model.to(config.device)
    # pce_over_seeds = np.array(pce_over_seeds)

/home/naomi/miniconda/lib/python3.12/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /home/naomi/miniconda/lib/python3.12/site-packages/i ...
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


TRAIN


AttributeError: 'float' object has no attribute 'requires_grad'

In [ ]:
x, y = next(iter(datamodule.val_dataloader()))
x = x.to(config.device)
y = y.to(config.device)
dist = model.predict(x)

In [ ]:
torch.norm(x[0])

tensor(5.8882, device='cuda:0')

In [ ]:
dist.mean.mean(dim=0)

tensor([ 0.0149,  0.0348, -0.0221], device='cuda:0', grad_fn=<MeanBackward1>)

In [ ]:
dist.mean.shape

torch.Size([256, 3])

In [ ]:
dist.covariance_matrix.mean(dim=0)

tensor([[ 0.0096,  0.0012, -0.0016],
        [ 0.0012,  0.0104, -0.0010],
        [-0.0016, -0.0010,  0.0094]], device='cuda:0', grad_fn=<MeanBackward1>)